# RQ1 Supplement: Single-stage Models under WSI-LOSO Global Threshold Calibration

## Experiment purpose

This notebook performs a **training-free, controlled decision-calibration experiment** for the formal RQ1 single-stage models:

1. Controlled Hard baseline
2. Label-aware sampler
3. Integrated Raw Soft
4. Integrated Soft cutoff 0.1
5. Integrated Soft cutoff 0.2
6. Integrated Soft cutoff 0.3

The experiment asks whether the weak fixed-threshold F1 of the Soft-label variants is mainly caused by a mismatch between their probability scale and the historical threshold of 0.5.

## Controlled design

All trained checkpoints are frozen. The Validation/Test hard labels, row order, model probabilities, threshold grid and metrics are shared. **Only the final global decision threshold changes.** One scalar threshold is applied to all 12 feature scores; no class-wise thresholds are used.

Three decision modes are compared:

- `Fixed_0.5`: historical reference.
- `LOSO_Global_MacroF1`: WSI-level LOSO selection that maximizes Macro F1.
- `LOSO_RecallPreserving_MacroF1`: maximizes Macro F1 while preserving both Micro Recall and Macro Recall relative to the Controlled Hard model at 0.5, with a pre-frozen tolerance of 0.005.

Thresholds are selected using Validation only. The final threshold is the median of six LOSO fold thresholds. Test data are not loaded until the threshold configuration has been written to an immutable freeze file.

## Frozen protocol and interpretation boundaries

- Primary threshold objective: Validation **Macro F1**.
- Ranking metrics: Micro/Macro AUROC and Micro/Macro AUPRC; these do not change with the threshold.
- Recall-preserving constraint is calculated against `Controlled_Hard @ 0.5` on the same five-WSI calibration subset in each LOSO fold.
- Threshold grid: 0.001–0.999 with step 0.001.
- Deterministic tie-break order: Macro F1 → Weighted F1 → Micro F1 → Macro Precision → closest to 0.5 → higher threshold.
- No model retraining, checkpoint replacement, per-class thresholding, temperature scaling, isotonic regression or Test-driven rule selection.
- Because this project has historically inspected the same Test split, any final Test output should be described as a **post-hoc confirmatory evaluation**, not a first blind Test.

In [1]:
from __future__ import annotations

import gc
import hashlib
import json
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score

# =============================================================================
# User switches
# =============================================================================

# Phase A should be run first with False. It freezes all thresholds from Validation.
# After the freeze file is successfully created, set this to True and run the Test phase.
RUN_TEST_EVALUATION = False

# Existing probabilities are always preferred. When Validation probabilities were
# not historically saved, this notebook may run inference from the frozen checkpoint.
ALLOW_INFERENCE_FALLBACK = True
RECOMPUTE_INFERENCE_CACHE = False

INFERENCE_BATCH_SIZE = 256
INFERENCE_NUM_WORKERS = 8

# =============================================================================
# Frozen scientific protocol
# =============================================================================

SEED = 42
THRESHOLD_MIN = 0.001
THRESHOLD_MAX = 0.999
THRESHOLD_STEP = 0.001
THRESHOLD_GRID = np.round(
    np.arange(THRESHOLD_MIN, THRESHOLD_MAX + THRESHOLD_STEP / 2, THRESHOLD_STEP),
    6,
)
RECALL_EPSILON = 0.005
EXPECTED_VAL_ROWS = 198_083
EXPECTED_TEST_ROWS = 314_284
EXPECTED_VAL_WSIS = 6
EXPECTED_TEST_WSIS = 6
EXPECTED_NUM_CLASSES = 12

LABEL_COLUMNS = [
    "Irregular epithelial stratification",
    "Loss of polarity of basal cells",
    "Drop shaped rete ridges",
    "Premature keratinization in single cells",
    "Loss of epithelial cell cohesion",
    "Abnormal variation in nuclear size",
    "Abnormal variation in nuclear shape",
    "Abnormal variation in cell size",
    "Abnormal variation in cell shape",
    "Increased N:C ratio",
    "Increased number and size of nucleoli",
    "Hyperchromasia",
]

MODEL_ORDER = [
    "Controlled_Hard",
    "LabelAware_Hard",
    "Integrated_RawSoft",
    "Integrated_SoftCutoff_0.1",
    "Integrated_SoftCutoff_0.2",
    "Integrated_SoftCutoff_0.3",
]

MODE_ORDER = [
    "Fixed_0.5",
    "LOSO_Global_MacroF1",
    "LOSO_RecallPreserving_MacroF1",
]

PROTOCOL = {
    "experiment": "RQ1 Single-stage Models under WSI-LOSO Global Threshold Calibration",
    "seed": SEED,
    "labels": LABEL_COLUMNS,
    "model_order": MODEL_ORDER,
    "mode_order": MODE_ORDER,
    "threshold_grid": {
        "min": THRESHOLD_MIN,
        "max": THRESHOLD_MAX,
        "step": THRESHOLD_STEP,
        "comparison": "probability >= threshold",
    },
    "global_f1_objective": "maximize validation hard-label Macro F1",
    "recall_preserving_constraints": {
        "reference": "Controlled_Hard at threshold 0.5 on the same LOSO calibration subset",
        "micro_recall_tolerance": RECALL_EPSILON,
        "macro_recall_tolerance": RECALL_EPSILON,
    },
    "tie_break_order": [
        "macro_f1 descending",
        "weighted_f1 descending",
        "micro_f1 descending",
        "macro_precision descending",
        "absolute distance from 0.5 ascending",
        "threshold descending",
    ],
    "final_threshold": "median of six WSI-LOSO fold thresholds",
    "test_policy": "load Test only after Validation thresholds are frozen",
}

PROTOCOL_JSON = json.dumps(PROTOCOL, sort_keys=True, ensure_ascii=False, separators=(",", ":"))
PROTOCOL_SHA256 = hashlib.sha256(PROTOCOL_JSON.encode("utf-8")).hexdigest()

print("Protocol SHA-256:", PROTOCOL_SHA256)
print("Threshold grid size:", len(THRESHOLD_GRID))
assert len(LABEL_COLUMNS) == EXPECTED_NUM_CLASSES
assert THRESHOLD_GRID[0] == THRESHOLD_MIN
assert THRESHOLD_GRID[-1] == THRESHOLD_MAX

Protocol SHA-256: b1c10fe3f08f9ca8a1c7296082bcec69c68bf2683d0001615a0a3a92a4188186
Threshold grid size: 999


In [2]:
# =============================================================================
# Project-root and historical-file resolution
# =============================================================================

ROOT_CANDIDATES = [
    Path(os.environ.get("JIANGJIE_PROJECT_ROOT", "")) if os.environ.get("JIANGJIE_PROJECT_ROOT") else None,
    Path("/scr/user/jiangjie/Jiangjie_Project"),
    Path("/home/user/jiangjie/Jiangjie_Project"),
    Path.cwd(),
]
ROOT_CANDIDATES = [p.resolve() for p in ROOT_CANDIDATES if p is not None]


def first_existing_root() -> Path:
    for root in ROOT_CANDIDATES:
        if (root / "data" / "ResearchProject_50" / "final_df_val.csv").exists():
            return root
    attempted = "\n".join(str(p) for p in ROOT_CANDIDATES)
    raise FileNotFoundError(
        "Could not locate Jiangjie_Project. Set environment variable "
        "JIANGJIE_PROJECT_ROOT or edit ROOT_CANDIDATES. Attempted:\n" + attempted
    )


PROJECT_ROOT = first_existing_root()
HARD_LABEL_DIR = PROJECT_ROOT / "data" / "ResearchProject_50"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "rq1_single_stage_wsi_loso_global_threshold_calibration"
CACHE_DIR = OUTPUT_DIR / "probability_cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def resolve_historical(relative_candidates: Sequence[str], required: bool = False) -> Optional[Path]:
    """Resolve a historical artifact across known project-root locations."""
    attempted: List[Path] = []
    for rel in relative_candidates:
        rel_path = Path(rel)
        if rel_path.is_absolute():
            attempted.append(rel_path)
            if rel_path.exists():
                return rel_path.resolve()
        else:
            for root in ROOT_CANDIDATES:
                candidate = root / rel_path
                attempted.append(candidate)
                if candidate.exists():
                    return candidate.resolve()
    if required:
        raise FileNotFoundError("Required artifact not found. Attempted:\n" + "\n".join(map(str, attempted)))
    return None


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


@dataclass(frozen=True)
class ModelSpec:
    model_id: str
    description: str
    checkpoint_candidates: Tuple[str, ...]
    val_probs_candidates: Tuple[str, ...] = ()
    val_targets_candidates: Tuple[str, ...] = ()
    test_probs_candidates: Tuple[str, ...] = ()
    test_targets_candidates: Tuple[str, ...] = ()


MODEL_SPECS: Dict[str, ModelSpec] = {
    "Controlled_Hard": ModelSpec(
        model_id="Controlled_Hard",
        description="50% Hard labels + dynamic 1:1 sampler + hard-derived sqrt BCE",
        checkpoint_candidates=(
            "outputs/kee_reproduction/controlled_seed42_50_sqrt_no_aug/best_model_Sqr_50_None.pth",
        ),
        val_probs_candidates=(
            "outputs/kee_reproduction/controlled_seed42_50_sqrt_no_aug/val_probs_Sqr_50_None.npy",
        ),
        val_targets_candidates=(
            "outputs/kee_reproduction/controlled_seed42_50_sqrt_no_aug/val_targets_hard_Sqr_50_None.npy",
        ),
        test_probs_candidates=(
            "outputs/kee_reproduction/controlled_seed42_50_sqrt_no_aug/test_probs_Sqr_50_None.npy",
        ),
        test_targets_candidates=(
            "outputs/kee_reproduction/controlled_seed42_50_sqrt_no_aug/test_targets_hard_Sqr_50_None.npy",
        ),
    ),
    "LabelAware_Hard": ModelSpec(
        model_id="LabelAware_Hard",
        description="Hard labels + label-aware balanced abnormal sampling",
        checkpoint_candidates=(
            "outputs/kee_reproduction/label_aware_balanced_sampler_seed42_50_sqrt_no_aug/best_model_Sqr_50_None.pth",
        ),
    ),
    "Integrated_RawSoft": ModelSpec(
        model_id="Integrated_RawSoft",
        description="Raw Soft target + Soft-aware 1:1 sampler + Soft-derived sqrt weight",
        checkpoint_candidates=(
            "outputs/soft_label_experiments/controlled_soft_raw_seed42_50_sqr_no_aug_aucselect/best_model_Sqr_SoftRaw_None_by_macro_auc.pth",
        ),
        test_probs_candidates=(
            "outputs/soft_label_experiments/controlled_soft_raw_seed42_50_sqr_no_aug_aucselect/test_probs_Sqr_SoftRaw_None.npy",
        ),
        test_targets_candidates=(
            "outputs/soft_label_experiments/controlled_soft_raw_seed42_50_sqr_no_aug_aucselect/test_targets_hard_Sqr_SoftRaw_None.npy",
        ),
    ),
    "Integrated_SoftCutoff_0.1": ModelSpec(
        model_id="Integrated_SoftCutoff_0.1",
        description="Soft cutoff 0.1 + cutoff-aware sampler and sqrt weight",
        checkpoint_candidates=(
            "outputs/soft_label_experiments/controlled_soft_cutoff01_seed42_50_sqr_no_aug_aucselect/best_model_Sqr_SoftCutoff01_None_by_macro_auc.pth",
        ),
        test_probs_candidates=(
            "outputs/soft_label_experiments/controlled_soft_cutoff01_seed42_50_sqr_no_aug_aucselect/test_probs_Sqr_SoftCutoff01_None.npy",
        ),
        test_targets_candidates=(
            "outputs/soft_label_experiments/controlled_soft_cutoff01_seed42_50_sqr_no_aug_aucselect/test_targets_hard_Sqr_SoftCutoff01_None.npy",
        ),
    ),
    "Integrated_SoftCutoff_0.2": ModelSpec(
        model_id="Integrated_SoftCutoff_0.2",
        description="Soft cutoff 0.2 + cutoff-aware sampler and sqrt weight",
        checkpoint_candidates=(
            "outputs/soft_label_experiments/controlled_soft_cutoff02_seed42_50_sqr_no_aug_aucselect/best_model_Sqr_SoftCutoff02_None_by_macro_auc.pth",
        ),
        test_probs_candidates=(
            "outputs/soft_label_experiments/controlled_soft_cutoff02_seed42_50_sqr_no_aug_aucselect/test_probs_Sqr_SoftCutoff02_None.npy",
        ),
        test_targets_candidates=(
            "outputs/soft_label_experiments/controlled_soft_cutoff02_seed42_50_sqr_no_aug_aucselect/test_targets_hard_Sqr_SoftCutoff02_None.npy",
        ),
    ),
    "Integrated_SoftCutoff_0.3": ModelSpec(
        model_id="Integrated_SoftCutoff_0.3",
        description="Soft cutoff 0.3 + cutoff-aware sampler and sqrt weight",
        checkpoint_candidates=(
            "outputs/soft_label_experiments/controlled_soft_cutoff03_seed42_50_sqr_no_aug_aucselect/best_model_Sqr_SoftCutoff03_None_by_macro_auc.pth",
        ),
        test_probs_candidates=(
            "outputs/soft_label_experiments/controlled_soft_cutoff03_seed42_50_sqr_no_aug_aucselect/test_probs_Sqr_SoftCutoff03_None.npy",
        ),
        test_targets_candidates=(
            "outputs/soft_label_experiments/controlled_soft_cutoff03_seed42_50_sqr_no_aug_aucselect/test_targets_hard_Sqr_SoftCutoff03_None.npy",
        ),
    ),
}

assert list(MODEL_SPECS) == MODEL_ORDER

print("PROJECT_ROOT:", PROJECT_ROOT)
print("HARD_LABEL_DIR:", HARD_LABEL_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

PROJECT_ROOT: /scr/user/jiangjie/Jiangjie_Project
HARD_LABEL_DIR: /scr/user/jiangjie/Jiangjie_Project/data/ResearchProject_50
OUTPUT_DIR: /scr/user/jiangjie/Jiangjie_Project/outputs/rq1_single_stage_wsi_loso_global_threshold_calibration


In [3]:
# =============================================================================
# Load Validation hard-label manifest only (Test remains untouched in Phase A)
# =============================================================================

VAL_CSV = HARD_LABEL_DIR / "final_df_val.csv"
assert VAL_CSV.exists(), VAL_CSV

val_df = pd.read_csv(VAL_CSV, low_memory=False)
missing_columns = [c for c in ["slide_name", "filepath", *LABEL_COLUMNS] if c not in val_df.columns]
assert not missing_columns, f"Validation CSV missing columns: {missing_columns}"

for col in LABEL_COLUMNS:
    val_df[col] = pd.to_numeric(val_df[col], errors="raise").astype(np.int8)

val_targets_csv = val_df[LABEL_COLUMNS].to_numpy(dtype=np.int8)
val_slide_ids = val_df["slide_name"].astype(str).to_numpy()
val_wsis = sorted(pd.unique(val_slide_ids).tolist())

assert len(val_df) == EXPECTED_VAL_ROWS, (len(val_df), EXPECTED_VAL_ROWS)
assert val_targets_csv.shape == (EXPECTED_VAL_ROWS, EXPECTED_NUM_CLASSES)
assert set(np.unique(val_targets_csv)).issubset({0, 1})
assert len(val_wsis) == EXPECTED_VAL_WSIS, (len(val_wsis), val_wsis)
assert not val_df.duplicated().any(), "Validation CSV contains duplicated complete rows."

manifest_key_columns = [c for c in ["slide_name", "filepath", "x", "y", "patch_id"] if c in val_df.columns]
manifest_payload = val_df[manifest_key_columns].astype(str).to_csv(index=False).encode("utf-8")
VAL_MANIFEST_SHA256 = hashlib.sha256(manifest_payload).hexdigest()

val_manifest_summary = pd.DataFrame({
    "item": ["rows", "WSIs", "positive_label_pairs", "manifest_sha256"],
    "value": [len(val_df), len(val_wsis), int(val_targets_csv.sum()), VAL_MANIFEST_SHA256],
})
display(val_manifest_summary)
print("Validation WSIs:", val_wsis)

,item,value
0,rows,198083
1,WSIs,6
2,positive_label_pairs,12031
3,manifest_sha256,b846e02703f569b1fe6a2ee071157572edd44e3ba12672...


Validation WSIs: ['1253-DP-12', '1318-DP-08', '1467-DP-12', '168-DP-09', '895-DP-09', 'D706-12-1-i']


In [4]:
# =============================================================================
# Frozen-checkpoint inference fallback
# =============================================================================

# Historical Controlled validation probabilities exist. The 06 AUC-selected
# Soft notebooks historically saved Test probabilities but not Validation
# probabilities; Label-aware saved neither. Missing arrays are therefore
# generated once from the frozen checkpoint and cached by this notebook.


def normalize_patch_path(raw_path: str) -> Path:
    raw = str(raw_path)
    direct = Path(raw)
    if direct.exists():
        return direct

    markers = ["ResearchProject_50/", "ResearchProject_50\\"]
    for marker in markers:
        if marker in raw:
            suffix = raw.split(marker, 1)[1].replace("\\", "/")
            candidate = HARD_LABEL_DIR / suffix
            if candidate.exists():
                return candidate

    old_prefixes = [
        "/content/ResearchProject_50",
        "/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50",
        "/scr/user/jiangjie/Jiangjie_Project/data/ResearchProject_50",
    ]
    for prefix in old_prefixes:
        if raw.startswith(prefix):
            candidate = Path(str(HARD_LABEL_DIR) + raw[len(prefix):])
            if candidate.exists():
                return candidate

    raise FileNotFoundError(f"Patch image not found after path normalization: {raw_path}")


def build_ctranspath_model(num_classes: int = EXPECTED_NUM_CLASSES):
    try:
        import torch
        import torch.nn as nn
        import timm
    except ImportError as exc:
        raise ImportError("Inference fallback requires torch and timm in the DICC environment.") from exc

    def to_2tuple_local(x):
        return tuple(x) if isinstance(x, (tuple, list)) else (x, x)

    class ConvStem(nn.Module):
        def __init__(self, img_size=224, patch_size=4, in_chans=3, embed_dim=768, norm_layer=None, flatten=True):
            super().__init__()
            assert patch_size == 4
            assert embed_dim % 8 == 0
            self.img_size = to_2tuple_local(img_size)
            self.patch_size = to_2tuple_local(patch_size)
            self.grid_size = (
                self.img_size[0] // self.patch_size[0],
                self.img_size[1] // self.patch_size[1],
            )
            self.num_patches = self.grid_size[0] * self.grid_size[1]
            self.flatten = flatten
            stem = []
            input_dim, output_dim = in_chans, embed_dim // 8
            for _ in range(2):
                stem.extend([
                    nn.Conv2d(input_dim, output_dim, kernel_size=3, stride=2, padding=1, bias=False),
                    nn.BatchNorm2d(output_dim),
                    nn.ReLU(inplace=True),
                ])
                input_dim = output_dim
                output_dim *= 2
            stem.append(nn.Conv2d(input_dim, embed_dim, kernel_size=1))
            self.proj = nn.Sequential(*stem)
            self.norm = norm_layer(embed_dim) if norm_layer else nn.Identity()

        def forward(self, x):
            _, _, height, width = x.shape
            assert (height, width) == self.img_size
            x = self.proj(x)
            if self.flatten:
                x = x.flatten(2).transpose(1, 2)
            return self.norm(x)

    model = timm.create_model(
        model_name="swin_tiny_patch4_window7_224",
        embed_layer=ConvStem,
        pretrained=False,
        num_classes=0,
    )
    model.global_pool = "avg"
    model.head = nn.Linear(model.num_features, num_classes)
    return model


def extract_state_dict(payload: Any) -> Mapping[str, Any]:
    if isinstance(payload, Mapping):
        for key in ("state_dict", "model_state_dict", "model"):
            value = payload.get(key)
            if isinstance(value, Mapping):
                return value
    if isinstance(payload, Mapping):
        return payload
    raise TypeError(f"Unsupported checkpoint payload type: {type(payload)}")


def infer_probabilities(checkpoint_path: Path, df: pd.DataFrame, split: str) -> np.ndarray:
    import torch
    from PIL import Image
    from torch.utils.data import DataLoader, Dataset
    from torchvision import transforms

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Inference fallback: {split}, checkpoint={checkpoint_path}, device={device}")

    transform = transforms.Compose([
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])

    class OrderedPatchDataset(Dataset):
        def __init__(self, frame: pd.DataFrame):
            self.paths = frame["filepath"].astype(str).tolist()

        def __len__(self):
            return len(self.paths)

        def __getitem__(self, index: int):
            path = normalize_patch_path(self.paths[index])
            try:
                with Image.open(path) as image:
                    image = image.convert("RGB")
                    tensor = transform(image)
            except Exception as exc:
                raise RuntimeError(f"Failed to load image at row {index}: {path}") from exc
            return tensor, index

    model = build_ctranspath_model().to(device)
    payload = torch.load(checkpoint_path, map_location="cpu")
    state_dict = extract_state_dict(payload)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    assert not missing, f"Missing checkpoint keys: {missing[:20]}"
    assert not unexpected, f"Unexpected checkpoint keys: {unexpected[:20]}"
    model.eval()

    dataset = OrderedPatchDataset(df)
    loader = DataLoader(
        dataset,
        batch_size=INFERENCE_BATCH_SIZE,
        shuffle=False,
        num_workers=INFERENCE_NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    probabilities = np.empty((len(dataset), EXPECTED_NUM_CLASSES), dtype=np.float32)
    seen = np.zeros(len(dataset), dtype=bool)

    with torch.inference_mode():
        for images, indices in loader:
            images = images.to(device, non_blocking=True)
            outputs = model(images)
            probs = torch.sigmoid(outputs).cpu().numpy().astype(np.float32)
            idx = indices.numpy().astype(np.int64)
            probabilities[idx] = probs
            seen[idx] = True

    assert seen.all(), f"Inference did not produce all rows: missing={int((~seen).sum())}"
    del model, loader, dataset
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return probabilities


def validate_probability_array(probs: np.ndarray, expected_rows: int, model_id: str, split: str) -> None:
    assert probs.shape == (expected_rows, EXPECTED_NUM_CLASSES), (
        model_id, split, probs.shape, (expected_rows, EXPECTED_NUM_CLASSES)
    )
    assert np.isfinite(probs).all(), f"Non-finite probabilities: {model_id} {split}"
    assert float(probs.min()) >= -1e-7, (model_id, split, float(probs.min()))
    assert float(probs.max()) <= 1.0 + 1e-7, (model_id, split, float(probs.max()))


def dataframe_signature(df: pd.DataFrame, expected_targets: np.ndarray) -> str:
    key_columns = [c for c in ["slide_name", "filepath", "x", "y", "patch_id"] if c in df.columns]
    digest = hashlib.sha256()
    digest.update(df[key_columns].astype(str).to_csv(index=False).encode("utf-8"))
    digest.update(np.asarray(expected_targets, dtype=np.int8).tobytes(order="C"))
    return digest.hexdigest()


def resolve_probabilities(
    spec: ModelSpec,
    split: str,
    df: pd.DataFrame,
    expected_targets: np.ndarray,
) -> Tuple[np.ndarray, Dict[str, Any]]:
    if split not in {"val", "test"}:
        raise ValueError(split)

    cache_probs = CACHE_DIR / f"{spec.model_id}_{split}_probs.npy"
    cache_targets = CACHE_DIR / f"{spec.model_id}_{split}_targets_hard.npy"
    cache_metadata = CACHE_DIR / f"{spec.model_id}_{split}_cache_metadata.json"

    probs_candidates = spec.val_probs_candidates if split == "val" else spec.test_probs_candidates
    targets_candidates = spec.val_targets_candidates if split == "val" else spec.test_targets_candidates

    historical_probs = resolve_historical(probs_candidates, required=False)
    historical_targets = resolve_historical(targets_candidates, required=False)
    checkpoint = resolve_historical(spec.checkpoint_candidates, required=True)
    checkpoint_sha = sha256_file(checkpoint)
    data_sha = dataframe_signature(df, expected_targets)

    expected_cache_signature = {
        "model_id": spec.model_id,
        "split": split,
        "checkpoint_path": str(checkpoint),
        "checkpoint_sha256": checkpoint_sha,
        "data_signature_sha256": data_sha,
        "rows": int(len(df)),
        "classes": EXPECTED_NUM_CLASSES,
    }

    cache_is_valid = False
    if cache_probs.exists() and cache_targets.exists() and cache_metadata.exists() and not RECOMPUTE_INFERENCE_CACHE:
        cached_meta = json.loads(cache_metadata.read_text(encoding="utf-8"))
        cache_is_valid = all(
            cached_meta.get(key) == value
            for key, value in expected_cache_signature.items()
        )
        if not cache_is_valid:
            print(f"Ignoring stale cache for {spec.model_id} {split}: provenance signature mismatch.")

    source_type: str
    source_path: Path
    if cache_is_valid:
        probs = np.load(cache_probs)
        cached_targets = np.load(cache_targets).astype(np.int8)
        assert np.array_equal(cached_targets, expected_targets)
        source_type = "verified_notebook_cache"
        source_path = cache_probs
    elif historical_probs is not None and not RECOMPUTE_INFERENCE_CACHE:
        probs = np.load(historical_probs)
        source_type = "historical_probability"
        source_path = historical_probs
    else:
        assert ALLOW_INFERENCE_FALLBACK, (
            f"No {split} probabilities found for {spec.model_id}, and inference fallback is disabled."
        )
        probs = infer_probabilities(checkpoint, df, split)
        source_type = "frozen_checkpoint_inference"
        source_path = checkpoint

    probs = np.asarray(probs, dtype=np.float32)
    validate_probability_array(probs, len(df), spec.model_id, split)

    if historical_targets is not None:
        saved_targets = np.load(historical_targets).astype(np.int8)
        assert saved_targets.shape == expected_targets.shape
        assert np.array_equal(saved_targets, expected_targets), (
            f"Saved targets do not match CSV hard labels: {spec.model_id} {split}"
        )

    # Cache all resolved arrays with a strict checkpoint/data provenance record.
    np.save(cache_probs, probs.astype(np.float32))
    np.save(cache_targets, expected_targets.astype(np.int8))
    cache_record = {
        **expected_cache_signature,
        "original_probability_source_type": source_type,
        "original_probability_source_path": str(source_path),
        "probability_sha256": sha256_file(cache_probs),
        "target_sha256": sha256_file(cache_targets),
    }
    cache_metadata.write_text(
        json.dumps(cache_record, indent=2, sort_keys=True),
        encoding="utf-8",
    )

    metadata = {
        "model_id": spec.model_id,
        "split": split,
        "probability_source_type": source_type,
        "probability_source_path": str(source_path),
        "cache_probability_path": str(cache_probs),
        "cache_metadata_path": str(cache_metadata),
        "checkpoint_path": str(checkpoint),
        "checkpoint_sha256": checkpoint_sha,
        "data_signature_sha256": data_sha,
        "rows": int(probs.shape[0]),
        "classes": int(probs.shape[1]),
        "probability_min": float(probs.min()),
        "probability_max": float(probs.max()),
        "probability_mean": float(probs.mean()),
    }
    return probs, metadata


In [5]:
# =============================================================================
# Resolve and audit all Validation probabilities
# =============================================================================

val_probs_by_model: Dict[str, np.ndarray] = {}
validation_input_audit_rows: List[Dict[str, Any]] = []

for model_id in MODEL_ORDER:
    print("\n" + "=" * 100)
    print("Resolving Validation probabilities:", model_id)
    probs, metadata = resolve_probabilities(
        MODEL_SPECS[model_id],
        split="val",
        df=val_df,
        expected_targets=val_targets_csv,
    )
    val_probs_by_model[model_id] = probs
    validation_input_audit_rows.append(metadata)

validation_input_audit_df = pd.DataFrame(validation_input_audit_rows)
validation_input_audit_df.to_csv(OUTPUT_DIR / "validation_input_audit.csv", index=False)
display(validation_input_audit_df)

assert list(val_probs_by_model) == MODEL_ORDER
assert all(arr.shape == val_targets_csv.shape for arr in val_probs_by_model.values())
print("PASS: all Validation arrays are finite, aligned and target-consistent.")


Resolving Validation probabilities: Controlled_Hard

Resolving Validation probabilities: LabelAware_Hard
Inference fallback: val, checkpoint=/scr/user/jiangjie/Jiangjie_Project/outputs/kee_reproduction/label_aware_balanced_sampler_seed42_50_sqrt_no_aug/best_model_Sqr_50_None.pth, device=cuda


/home/user/jiangjie/.conda/envs/jiangjie/lib/python3.10/site-packages/torch/functional.py:539: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:3637.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]



Resolving Validation probabilities: Integrated_RawSoft
Inference fallback: val, checkpoint=/scr/user/jiangjie/Jiangjie_Project/outputs/soft_label_experiments/controlled_soft_raw_seed42_50_sqr_no_aug_aucselect/best_model_Sqr_SoftRaw_None_by_macro_auc.pth, device=cuda

Resolving Validation probabilities: Integrated_SoftCutoff_0.1
Inference fallback: val, checkpoint=/scr/user/jiangjie/Jiangjie_Project/outputs/soft_label_experiments/controlled_soft_cutoff01_seed42_50_sqr_no_aug_aucselect/best_model_Sqr_SoftCutoff01_None_by_macro_auc.pth, device=cuda

Resolving Validation probabilities: Integrated_SoftCutoff_0.2
Inference fallback: val, checkpoint=/scr/user/jiangjie/Jiangjie_Project/outputs/soft_label_experiments/controlled_soft_cutoff02_seed42_50_sqr_no_aug_aucselect/best_model_Sqr_SoftCutoff02_None_by_macro_auc.pth, device=cuda

Resolving Validation probabilities: Integrated_SoftCutoff_0.3
Inference fallback: val, checkpoint=/scr/user/jiangjie/Jiangjie_Project/outputs/soft_label_experime

,model_id,split,probability_source_type,probability_source_path,cache_probability_path,cache_metadata_path,checkpoint_path,checkpoint_sha256,data_signature_sha256,rows,classes,probability_min,probability_max,probability_mean
0,Controlled_Hard,val,historical_probability,/scr/user/jiangjie/Jiangjie_Project/outputs/ke...,/scr/user/jiangjie/Jiangjie_Project/outputs/rq...,/scr/user/jiangjie/Jiangjie_Project/outputs/rq...,/scr/user/jiangjie/Jiangjie_Project/outputs/ke...,972c6474bacfd98ad66daaaad1a4fdfbec0c76ca566963...,a4f6317c829ee9288911230bd811eb35eb8e53d34632ff...,198083,12,0.000056,0.995071,0.092776
1,LabelAware_Hard,val,frozen_checkpoint_inference,/scr/user/jiangjie/Jiangjie_Project/outputs/ke...,/scr/user/jiangjie/Jiangjie_Project/outputs/rq...,/scr/user/jiangjie/Jiangjie_Project/outputs/rq...,/scr/user/jiangjie/Jiangjie_Project/outputs/ke...,403614d3bead98d662c1c5c934faef0af9d563c1601c2b...,a4f6317c829ee9288911230bd811eb35eb8e53d34632ff...,198083,12,0.000091,0.998264,0.110632
2,Integrated_RawSoft,val,frozen_checkpoint_inference,/scr/user/jiangjie/Jiangjie_Project/outputs/so...,/scr/user/jiangjie/Jiangjie_Project/outputs/rq...,/scr/user/jiangjie/Jiangjie_Project/outputs/rq...,/scr/user/jiangjie/Jiangjie_Project/outputs/so...,02d92dfe04a39eb2319b404571676c370f37a803c2777c...,a4f6317c829ee9288911230bd811eb35eb8e53d34632ff...,198083,12,0.000109,0.992410,0.109685
3,Integrated_SoftCutoff_0.1,val,frozen_checkpoint_inference,/scr/user/jiangjie/Jiangjie_Project/outputs/so...,/scr/user/jiangjie/Jiangjie_Project/outputs/rq...,/scr/user/jiangjie/Jiangjie_Project/outputs/rq...,/scr/user/jiangjie/Jiangjie_Project/outputs/so...,6645f0b0fc4ef5dbdbe57ef058bf05bfb848ab14e57e47...,a4f6317c829ee9288911230bd811eb35eb8e53d34632ff...,198083,12,0.000024,0.994438,0.088230
4,Integrated_SoftCutoff_0.2,val,frozen_checkpoint_inference,/scr/user/jiangjie/Jiangjie_Project/outputs/so...,/scr/user/jiangjie/Jiangjie_Project/outputs/rq...,/scr/user/jiangjie/Jiangjie_Project/outputs/rq...,/scr/user/jiangjie/Jiangjie_Project/outputs/so...,1199fb9cab31f20fdfa0f8bc0679d9a8184716f9cf4862...,a4f6317c829ee9288911230bd811eb35eb8e53d34632ff...,198083,12,0.001886,0.916988,0.141766
5,Integrated_SoftCutoff_0.3,val,frozen_checkpoint_inference,/scr/user/jiangjie/Jiangjie_Project/outputs/so...,/scr/user/jiangjie/Jiangjie_Project/outputs/rq...,/scr/user/jiangjie/Jiangjie_Project/outputs/rq...,/scr/user/jiangjie/Jiangjie_Project/outputs/so...,65bdba88437df566330f5f1fa843621f81924f592b4457...,a4f6317c829ee9288911230bd811eb35eb8e53d34632ff...,198083,12,0.000114,0.997188,0.084698


PASS: all Validation arrays are finite, aligned and target-consistent.


In [6]:
# =============================================================================
# Metric and efficient global-threshold utilities
# =============================================================================


def safe_divide(numerator: np.ndarray | float, denominator: np.ndarray | float) -> np.ndarray:
    numerator_arr = np.asarray(numerator, dtype=np.float64)
    denominator_arr = np.asarray(denominator, dtype=np.float64)
    return np.divide(
        numerator_arr,
        denominator_arr,
        out=np.zeros_like(numerator_arr, dtype=np.float64),
        where=denominator_arr != 0,
    )


def ranking_metrics(y_true: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    y_true = np.asarray(y_true, dtype=np.int8)
    y_prob = np.asarray(y_prob, dtype=np.float64)

    micro_auroc = float(roc_auc_score(y_true.ravel(), y_prob.ravel()))
    micro_auprc = float(average_precision_score(y_true.ravel(), y_prob.ravel()))

    per_class_auroc: List[float] = []
    per_class_auprc: List[float] = []
    for class_index in range(y_true.shape[1]):
        labels = y_true[:, class_index]
        scores = y_prob[:, class_index]
        if np.unique(labels).size < 2:
            per_class_auroc.append(np.nan)
        else:
            per_class_auroc.append(float(roc_auc_score(labels, scores)))
        if labels.sum() == 0:
            per_class_auprc.append(np.nan)
        else:
            per_class_auprc.append(float(average_precision_score(labels, scores)))

    return {
        "micro_auroc": micro_auroc,
        "macro_auroc": float(np.nanmean(per_class_auroc)),
        "micro_auprc": micro_auprc,
        "macro_auprc": float(np.nanmean(per_class_auprc)),
    }


def count_metrics_from_pred(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float | int]:
    y_true = np.asarray(y_true, dtype=np.int8)
    y_pred = np.asarray(y_pred, dtype=np.int8)
    assert y_true.shape == y_pred.shape

    tp_c = ((y_true == 1) & (y_pred == 1)).sum(axis=0).astype(np.int64)
    fp_c = ((y_true == 0) & (y_pred == 1)).sum(axis=0).astype(np.int64)
    fn_c = ((y_true == 1) & (y_pred == 0)).sum(axis=0).astype(np.int64)
    support_c = tp_c + fn_c

    precision_c = safe_divide(tp_c, tp_c + fp_c)
    recall_c = safe_divide(tp_c, support_c)
    f1_c = safe_divide(2 * precision_c * recall_c, precision_c + recall_c)

    tp = int(tp_c.sum())
    fp = int(fp_c.sum())
    fn = int(fn_c.sum())
    support = int(support_c.sum())

    micro_precision = float(safe_divide(tp, tp + fp))
    micro_recall = float(safe_divide(tp, support))
    micro_f1 = float(safe_divide(2 * micro_precision * micro_recall, micro_precision + micro_recall))

    return {
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_precision": float(np.mean(precision_c)),
        "macro_recall": float(np.mean(recall_c)),
        "macro_f1": float(np.mean(f1_c)),
        "weighted_f1": float(np.average(f1_c, weights=support_c)) if support > 0 else 0.0,
        "true_positive_label_count": tp,
        "false_positive_label_count": fp,
        "false_negative_label_count": fn,
        "predicted_positive_label_count": int(y_pred.sum()),
        "ground_truth_positive_label_count": support,
    }


def evaluate_predictions(y_true: np.ndarray, y_prob: np.ndarray, y_pred: np.ndarray) -> Dict[str, float | int]:
    return {**ranking_metrics(y_true, y_prob), **count_metrics_from_pred(y_true, y_pred)}


def evaluate_at_threshold(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> Dict[str, float | int]:
    y_pred = (np.asarray(y_prob) >= float(threshold)).astype(np.int8)
    return {"threshold": float(threshold), **evaluate_predictions(y_true, y_prob, y_pred)}


def per_class_metrics(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    y_pred: np.ndarray,
    label_columns: Sequence[str] = LABEL_COLUMNS,
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    for class_index, label in enumerate(label_columns):
        labels = y_true[:, class_index].astype(np.int8)
        scores = y_prob[:, class_index].astype(np.float64)
        preds = y_pred[:, class_index].astype(np.int8)
        tp = int(((labels == 1) & (preds == 1)).sum())
        fp = int(((labels == 0) & (preds == 1)).sum())
        fn = int(((labels == 1) & (preds == 0)).sum())
        precision = float(safe_divide(tp, tp + fp))
        recall = float(safe_divide(tp, tp + fn))
        f1 = float(safe_divide(2 * precision * recall, precision + recall))
        rows.append({
            "class_index": class_index,
            "feature": label,
            "support": int(labels.sum()),
            "auroc": float(roc_auc_score(labels, scores)) if np.unique(labels).size == 2 else np.nan,
            "auprc": float(average_precision_score(labels, scores)) if labels.sum() > 0 else np.nan,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "predicted_positive": int(preds.sum()),
        })
    return pd.DataFrame(rows)


def global_threshold_curve(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    thresholds: np.ndarray = THRESHOLD_GRID,
) -> pd.DataFrame:
    """Efficiently calculate global-threshold metrics using sorted score arrays."""
    y_true = np.asarray(y_true, dtype=np.int8)
    y_prob = np.asarray(y_prob, dtype=np.float64)
    thresholds = np.asarray(thresholds, dtype=np.float64)
    n_thresholds = len(thresholds)
    n_classes = y_true.shape[1]

    tp = np.zeros((n_thresholds, n_classes), dtype=np.int64)
    fp = np.zeros((n_thresholds, n_classes), dtype=np.int64)
    support = y_true.sum(axis=0).astype(np.int64)

    for class_index in range(n_classes):
        scores = y_prob[:, class_index]
        labels = y_true[:, class_index]
        positive_scores = np.sort(scores[labels == 1])
        negative_scores = np.sort(scores[labels == 0])
        tp[:, class_index] = len(positive_scores) - np.searchsorted(
            positive_scores, thresholds, side="left"
        )
        fp[:, class_index] = len(negative_scores) - np.searchsorted(
            negative_scores, thresholds, side="left"
        )

    fn = support.reshape(1, -1) - tp
    precision_c = safe_divide(tp, tp + fp)
    recall_c = safe_divide(tp, support.reshape(1, -1))
    f1_c = safe_divide(2 * precision_c * recall_c, precision_c + recall_c)

    tp_micro = tp.sum(axis=1)
    fp_micro = fp.sum(axis=1)
    fn_micro = fn.sum(axis=1)
    total_support = int(support.sum())

    micro_precision = safe_divide(tp_micro, tp_micro + fp_micro)
    micro_recall = safe_divide(tp_micro, tp_micro + fn_micro)
    micro_f1 = safe_divide(2 * micro_precision * micro_recall, micro_precision + micro_recall)

    weighted_f1 = (f1_c * support.reshape(1, -1)).sum(axis=1) / max(total_support, 1)

    return pd.DataFrame({
        "threshold": thresholds,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_precision": precision_c.mean(axis=1),
        "macro_recall": recall_c.mean(axis=1),
        "macro_f1": f1_c.mean(axis=1),
        "weighted_f1": weighted_f1,
        "true_positive_label_count": tp_micro,
        "false_positive_label_count": fp_micro,
        "false_negative_label_count": fn_micro,
        "predicted_positive_label_count": (tp_micro + fp_micro),
        "ground_truth_positive_label_count": total_support,
    })


def select_best_threshold(
    curve: pd.DataFrame,
    required_micro_recall: Optional[float] = None,
    required_macro_recall: Optional[float] = None,
) -> Tuple[float, pd.Series, int]:
    eligible = curve.copy()
    if required_micro_recall is not None:
        eligible = eligible[eligible["micro_recall"] >= required_micro_recall - 1e-12]
    if required_macro_recall is not None:
        eligible = eligible[eligible["macro_recall"] >= required_macro_recall - 1e-12]
    if eligible.empty:
        raise RuntimeError(
            "No threshold satisfies the pre-frozen recall constraints. "
            "This should not occur because the grid includes very low thresholds."
        )

    eligible = eligible.assign(distance_to_0_5=(eligible["threshold"] - 0.5).abs())
    ranked = eligible.sort_values(
        by=[
            "macro_f1",
            "weighted_f1",
            "micro_f1",
            "macro_precision",
            "distance_to_0_5",
            "threshold",
        ],
        ascending=[False, False, False, False, True, False],
        kind="mergesort",
    )
    winner = ranked.iloc[0]
    return float(winner["threshold"]), winner, int(len(eligible))


def threshold_summary(values: Sequence[float]) -> Dict[str, float]:
    arr = np.asarray(values, dtype=np.float64)
    q1 = float(np.quantile(arr, 0.25))
    q3 = float(np.quantile(arr, 0.75))
    return {
        "final_threshold_median": float(np.median(arr)),
        "threshold_q1": q1,
        "threshold_q3": q3,
        "threshold_iqr": q3 - q1,
        "threshold_min": float(arr.min()),
        "threshold_max": float(arr.max()),
        "threshold_mean": float(arr.mean()),
        "threshold_sd": float(arr.std(ddof=0)),
    }

In [7]:
# =============================================================================
# Internal synthetic protocol tests (fast; no project data)
# =============================================================================

rng = np.random.default_rng(123)
synthetic_y = rng.integers(0, 2, size=(200, 3), dtype=np.int8)
# Ensure both labels exist in every class.
synthetic_y[:3] = np.eye(3, dtype=np.int8)
synthetic_y[3:6] = 0
synthetic_p = np.clip(0.15 + 0.65 * synthetic_y + rng.normal(0, 0.15, size=synthetic_y.shape), 0, 1)
synthetic_thresholds = np.array([0.2, 0.5, 0.8], dtype=float)

synthetic_curve = global_threshold_curve(synthetic_y, synthetic_p, synthetic_thresholds)
for threshold in synthetic_thresholds:
    direct = evaluate_at_threshold(synthetic_y, synthetic_p, threshold)
    row = synthetic_curve.loc[np.isclose(synthetic_curve["threshold"], threshold)].iloc[0]
    for metric in [
        "micro_precision", "micro_recall", "micro_f1",
        "macro_precision", "macro_recall", "macro_f1", "weighted_f1",
        "true_positive_label_count", "false_positive_label_count", "false_negative_label_count",
    ]:
        assert np.isclose(float(row[metric]), float(direct[metric]), atol=1e-12), (threshold, metric)

ranking_before = ranking_metrics(synthetic_y, synthetic_p)
_ = (synthetic_p >= 0.2).astype(np.int8)
ranking_after = ranking_metrics(synthetic_y, synthetic_p)
assert ranking_before == ranking_after, "Thresholding must not change AUROC/AUPRC inputs."

base = evaluate_at_threshold(synthetic_y, synthetic_p, 0.5)
selected_t, selected_row, feasible_count = select_best_threshold(
    global_threshold_curve(synthetic_y, synthetic_p),
    required_micro_recall=max(0.0, float(base["micro_recall"]) - RECALL_EPSILON),
    required_macro_recall=max(0.0, float(base["macro_recall"]) - RECALL_EPSILON),
)
assert selected_row["micro_recall"] >= float(base["micro_recall"]) - RECALL_EPSILON - 1e-12
assert selected_row["macro_recall"] >= float(base["macro_recall"]) - RECALL_EPSILON - 1e-12
assert feasible_count > 0

print("PASS: synthetic metric, threshold-curve and recall-constraint tests.")
print("Synthetic recall-preserving threshold:", selected_t)

PASS: synthetic metric, threshold-curve and recall-constraint tests.
Synthetic recall-preserving threshold: 0.435


In [8]:
# =============================================================================
# Phase A: six-fold WSI-LOSO threshold calibration on Validation
# =============================================================================

fold_threshold_rows: List[Dict[str, Any]] = []
threshold_curve_rows: List[pd.DataFrame] = []
threshold_stability_rows: List[Dict[str, Any]] = []
validation_oof_rows: List[Dict[str, Any]] = []
validation_frozen_rows: List[Dict[str, Any]] = []
validation_oof_per_class_frames: List[pd.DataFrame] = []
validation_frozen_per_class_frames: List[pd.DataFrame] = []
validation_oof_per_wsi_rows: List[Dict[str, Any]] = []

frozen_thresholds: Dict[str, Dict[str, float]] = {}
fold_thresholds_nested: Dict[str, Dict[str, List[float]]] = {}

controlled_val_probs = val_probs_by_model["Controlled_Hard"]

for model_id in MODEL_ORDER:
    print("\n" + "=" * 110)
    print("WSI-LOSO calibration:", model_id)
    model_probs = val_probs_by_model[model_id]

    oof_predictions = {
        mode: np.zeros_like(val_targets_csv, dtype=np.int8)
        for mode in MODE_ORDER
    }
    model_fold_thresholds = {
        "Fixed_0.5": [],
        "LOSO_Global_MacroF1": [],
        "LOSO_RecallPreserving_MacroF1": [],
    }

    for fold_index, heldout_wsi in enumerate(val_wsis):
        heldout_mask = val_slide_ids == heldout_wsi
        calibration_mask = ~heldout_mask
        assert heldout_mask.any() and calibration_mask.any()

        y_cal = val_targets_csv[calibration_mask]
        p_cal = model_probs[calibration_mask]
        controlled_p_cal = controlled_val_probs[calibration_mask]

        curve = global_threshold_curve(y_cal, p_cal)
        curve_export = curve.copy()
        curve_export.insert(0, "heldout_wsi", heldout_wsi)
        curve_export.insert(0, "fold_index", fold_index)
        curve_export.insert(0, "model_id", model_id)
        threshold_curve_rows.append(curve_export)

        controlled_reference = evaluate_at_threshold(y_cal, controlled_p_cal, 0.5)
        required_micro = max(0.0, float(controlled_reference["micro_recall"]) - RECALL_EPSILON)
        required_macro = max(0.0, float(controlled_reference["macro_recall"]) - RECALL_EPSILON)

        f1_threshold, f1_winner, f1_eligible_count = select_best_threshold(curve)
        recall_threshold, recall_winner, recall_eligible_count = select_best_threshold(
            curve,
            required_micro_recall=required_micro,
            required_macro_recall=required_macro,
        )

        selected_by_mode = {
            "Fixed_0.5": (0.5, None, len(curve)),
            "LOSO_Global_MacroF1": (f1_threshold, f1_winner, f1_eligible_count),
            "LOSO_RecallPreserving_MacroF1": (
                recall_threshold,
                recall_winner,
                recall_eligible_count,
            ),
        }

        for mode, (threshold, winner, eligible_count) in selected_by_mode.items():
            model_fold_thresholds[mode].append(float(threshold))
            oof_predictions[mode][heldout_mask] = (
                model_probs[heldout_mask] >= float(threshold)
            ).astype(np.int8)

            fold_row: Dict[str, Any] = {
                "model_id": model_id,
                "mode": mode,
                "fold_index": fold_index,
                "heldout_wsi": heldout_wsi,
                "calibration_rows": int(calibration_mask.sum()),
                "heldout_rows": int(heldout_mask.sum()),
                "selected_threshold": float(threshold),
                "eligible_threshold_count": int(eligible_count),
                "controlled_reference_micro_recall": float(controlled_reference["micro_recall"]),
                "controlled_reference_macro_recall": float(controlled_reference["macro_recall"]),
                "required_micro_recall": float(required_micro),
                "required_macro_recall": float(required_macro),
            }
            if winner is not None:
                for metric in [
                    "micro_precision", "micro_recall", "micro_f1",
                    "macro_precision", "macro_recall", "macro_f1", "weighted_f1",
                    "true_positive_label_count", "false_positive_label_count",
                    "false_negative_label_count", "predicted_positive_label_count",
                ]:
                    fold_row[f"calibration_{metric}"] = float(winner[metric])
            fold_threshold_rows.append(fold_row)

    frozen_thresholds[model_id] = {}
    fold_thresholds_nested[model_id] = {}

    for mode in MODE_ORDER:
        thresholds = model_fold_thresholds[mode]
        stats = threshold_summary(thresholds)
        final_threshold = float(stats["final_threshold_median"])
        frozen_thresholds[model_id][mode] = final_threshold
        fold_thresholds_nested[model_id][mode] = [float(x) for x in thresholds]

        threshold_stability_rows.append({
            "model_id": model_id,
            "mode": mode,
            **stats,
        })

        # LOSO OOF evaluation: each WSI uses the threshold selected without that WSI.
        oof_pred = oof_predictions[mode]
        oof_metrics = evaluate_predictions(val_targets_csv, model_probs, oof_pred)
        validation_oof_rows.append({
            "model_id": model_id,
            "mode": mode,
            "evaluation": "Validation_LOSO_OOF",
            "reported_threshold": np.nan if mode != "Fixed_0.5" else 0.5,
            **oof_metrics,
        })

        oof_per_class = per_class_metrics(val_targets_csv, model_probs, oof_pred)
        oof_per_class.insert(0, "evaluation", "Validation_LOSO_OOF")
        oof_per_class.insert(0, "mode", mode)
        oof_per_class.insert(0, "model_id", model_id)
        validation_oof_per_class_frames.append(oof_per_class)

        for wsi in val_wsis:
            mask = val_slide_ids == wsi
            wsi_metrics = evaluate_predictions(
                val_targets_csv[mask], model_probs[mask], oof_pred[mask]
            )
            validation_oof_per_wsi_rows.append({
                "model_id": model_id,
                "mode": mode,
                "evaluation": "Validation_LOSO_OOF",
                "wsi": wsi,
                "rows": int(mask.sum()),
                **wsi_metrics,
            })

        # Frozen median threshold on the complete Validation split.
        frozen_pred = (model_probs >= final_threshold).astype(np.int8)
        frozen_metrics = evaluate_predictions(val_targets_csv, model_probs, frozen_pred)
        validation_frozen_rows.append({
            "model_id": model_id,
            "mode": mode,
            "evaluation": "Validation_FrozenMedianThreshold",
            "reported_threshold": final_threshold,
            **frozen_metrics,
        })

        frozen_per_class = per_class_metrics(val_targets_csv, model_probs, frozen_pred)
        frozen_per_class.insert(0, "evaluation", "Validation_FrozenMedianThreshold")
        frozen_per_class.insert(0, "mode", mode)
        frozen_per_class.insert(0, "model_id", model_id)
        validation_frozen_per_class_frames.append(frozen_per_class)

fold_threshold_df = pd.DataFrame(fold_threshold_rows)
threshold_curve_df = pd.concat(threshold_curve_rows, ignore_index=True)
threshold_stability_df = pd.DataFrame(threshold_stability_rows)
validation_oof_df = pd.DataFrame(validation_oof_rows)
validation_frozen_df = pd.DataFrame(validation_frozen_rows)
validation_oof_per_class_df = pd.concat(validation_oof_per_class_frames, ignore_index=True)
validation_frozen_per_class_df = pd.concat(validation_frozen_per_class_frames, ignore_index=True)
validation_oof_per_wsi_df = pd.DataFrame(validation_oof_per_wsi_rows)

fold_threshold_df.to_csv(OUTPUT_DIR / "validation_loso_fold_thresholds.csv", index=False)
threshold_curve_df.to_csv(OUTPUT_DIR / "validation_threshold_sweep_curves.csv", index=False)
threshold_stability_df.to_csv(OUTPUT_DIR / "validation_threshold_stability.csv", index=False)
validation_oof_df.to_csv(OUTPUT_DIR / "validation_loso_oof_overall.csv", index=False)
validation_frozen_df.to_csv(OUTPUT_DIR / "validation_frozen_median_overall.csv", index=False)
validation_oof_per_class_df.to_csv(OUTPUT_DIR / "validation_loso_oof_per_class.csv", index=False)
validation_frozen_per_class_df.to_csv(OUTPUT_DIR / "validation_frozen_median_per_class.csv", index=False)
validation_oof_per_wsi_df.to_csv(OUTPUT_DIR / "validation_loso_oof_per_wsi.csv", index=False)

print("PASS: all six models completed six-fold WSI-LOSO calibration.")


WSI-LOSO calibration: Controlled_Hard

WSI-LOSO calibration: LabelAware_Hard

WSI-LOSO calibration: Integrated_RawSoft

WSI-LOSO calibration: Integrated_SoftCutoff_0.1

WSI-LOSO calibration: Integrated_SoftCutoff_0.2

WSI-LOSO calibration: Integrated_SoftCutoff_0.3
PASS: all six models completed six-fold WSI-LOSO calibration.


In [9]:
# =============================================================================
# Validation review and immutable threshold freeze
# =============================================================================

baseline_val = validation_frozen_df[
    (validation_frozen_df["model_id"] == "Controlled_Hard")
    & (validation_frozen_df["mode"] == "Fixed_0.5")
].iloc[0]

validation_comparison_df = validation_frozen_df.copy()
for metric in [
    "micro_auroc", "macro_auroc", "micro_auprc", "macro_auprc",
    "micro_precision", "micro_recall", "micro_f1",
    "macro_precision", "macro_recall", "macro_f1", "weighted_f1",
]:
    validation_comparison_df[f"delta_vs_controlled_fixed_{metric}"] = (
        validation_comparison_df[metric] - float(baseline_val[metric])
    )

# Direct within-model threshold effect on Validation.
fixed_val_by_model = validation_frozen_df[
    validation_frozen_df["mode"] == "Fixed_0.5"
].set_index("model_id")
validation_threshold_effect_rows = []
for _, row in validation_frozen_df.iterrows():
    fixed_row = fixed_val_by_model.loc[row["model_id"]]
    effect = {
        "model_id": row["model_id"],
        "mode": row["mode"],
        "reported_threshold": row["reported_threshold"],
    }
    for metric in [
        "micro_precision", "micro_recall", "micro_f1",
        "macro_precision", "macro_recall", "macro_f1", "weighted_f1",
        "true_positive_label_count", "false_positive_label_count",
        "false_negative_label_count", "predicted_positive_label_count",
    ]:
        effect[f"delta_vs_same_model_fixed05_{metric}"] = float(row[metric]) - float(fixed_row[metric])
    validation_threshold_effect_rows.append(effect)
validation_threshold_effect_df = pd.DataFrame(validation_threshold_effect_rows)
validation_threshold_effect_df.to_csv(
    OUTPUT_DIR / "validation_threshold_effect_vs_same_model_fixed05.csv", index=False
)

# Audit whether the final median threshold still preserves Controlled recall on full Validation.
validation_comparison_df["micro_recall_preserved_vs_controlled"] = (
    validation_comparison_df["micro_recall"] >= float(baseline_val["micro_recall"]) - RECALL_EPSILON
)
validation_comparison_df["macro_recall_preserved_vs_controlled"] = (
    validation_comparison_df["macro_recall"] >= float(baseline_val["macro_recall"]) - RECALL_EPSILON
)
validation_comparison_df["both_recalls_preserved_vs_controlled"] = (
    validation_comparison_df["micro_recall_preserved_vs_controlled"]
    & validation_comparison_df["macro_recall_preserved_vs_controlled"]
)

validation_comparison_df.to_csv(
    OUTPUT_DIR / "validation_frozen_comparison_to_controlled_fixed.csv", index=False
)

freeze_payload = {
    "protocol_sha256": PROTOCOL_SHA256,
    "protocol": PROTOCOL,
    "validation_manifest_sha256": VAL_MANIFEST_SHA256,
    "validation_rows": int(len(val_df)),
    "validation_wsis": val_wsis,
    "model_checkpoint_sha256": {
        row["model_id"]: row["checkpoint_sha256"]
        for row in validation_input_audit_rows
    },
    "frozen_thresholds": frozen_thresholds,
    "fold_thresholds": fold_thresholds_nested,
    "threshold_stability": threshold_stability_df.to_dict(orient="records"),
}

FREEZE_PATH = OUTPUT_DIR / "frozen_validation_thresholds.json"
FREEZE_SHA_PATH = OUTPUT_DIR / "frozen_validation_thresholds.sha256"
freeze_text = json.dumps(freeze_payload, indent=2, ensure_ascii=False, sort_keys=True)
freeze_sha = hashlib.sha256(freeze_text.encode("utf-8")).hexdigest()

if FREEZE_PATH.exists():
    existing = json.loads(FREEZE_PATH.read_text(encoding="utf-8"))
    assert existing.get("protocol_sha256") == PROTOCOL_SHA256, (
        "An existing freeze file uses a different protocol. Do not overwrite it silently. "
        "Use a new OUTPUT_DIR or explicitly archive the old file."
    )
    assert existing.get("frozen_thresholds") == freeze_payload["frozen_thresholds"], (
        "Recomputed thresholds differ from the existing frozen thresholds. "
        "Stop and audit probability/checkpoint inputs before Test evaluation."
    )
    print("Existing freeze file verified; thresholds are unchanged.")
else:
    FREEZE_PATH.write_text(freeze_text, encoding="utf-8")
    FREEZE_SHA_PATH.write_text(freeze_sha + "\n", encoding="utf-8")
    print("Created immutable Validation threshold freeze:", FREEZE_PATH)

print("Freeze SHA-256:", freeze_sha)

summary_columns = [
    "model_id", "mode", "reported_threshold",
    "micro_auroc", "macro_auroc", "micro_auprc", "macro_auprc",
    "micro_precision", "micro_recall", "micro_f1",
    "macro_precision", "macro_recall", "macro_f1", "weighted_f1",
]
display(validation_frozen_df[summary_columns].sort_values(["model_id", "mode"]))
display(threshold_stability_df.sort_values(["model_id", "mode"]))
display(validation_threshold_effect_df.sort_values(["model_id", "mode"]))

print("Phase A complete. Thresholds are frozen from Validation only.")
print("To run Phase B, set RUN_TEST_EVALUATION=True without changing the protocol.")

Created immutable Validation threshold freeze: /scr/user/jiangjie/Jiangjie_Project/outputs/rq1_single_stage_wsi_loso_global_threshold_calibration/frozen_validation_thresholds.json
Freeze SHA-256: 0cdfa31aa74d8d00c05e075129bdecd08e76d89448534e29e066439e921d711d


,model_id,mode,reported_threshold,micro_auroc,macro_auroc,micro_auprc,macro_auprc,micro_precision,micro_recall,micro_f1,macro_precision,macro_recall,macro_f1,weighted_f1
0,Controlled_Hard,Fixed_0.5,0.5000,0.908747,0.889874,0.080729,0.067012,0.054389,0.565123,0.099228,0.056341,0.441474,0.093465,0.138141
1,Controlled_Hard,LOSO_Global_MacroF1,0.6685,0.908747,0.889874,0.080729,0.067012,0.082581,0.402460,0.137043,0.072742,0.299463,0.109427,0.163698
2,Controlled_Hard,LOSO_RecallPreserving_MacroF1,0.5050,0.908747,0.889874,0.080729,0.067012,0.055119,0.561051,0.100376,0.056939,0.437675,0.094188,0.139289
6,Integrated_RawSoft,Fixed_0.5,0.5000,0.892514,0.878513,0.053317,0.051649,0.041851,0.553986,0.077823,0.033466,0.427906,0.061001,0.091688
7,Integrated_RawSoft,LOSO_Global_MacroF1,0.7340,0.892514,0.878513,0.053317,0.051649,0.069309,0.245782,0.108127,0.057800,0.169716,0.081768,0.123887
8,Integrated_RawSoft,LOSO_RecallPreserving_MacroF1,0.4870,0.892514,0.878513,0.053317,0.051649,0.040780,0.569944,0.076114,0.032672,0.443827,0.059836,0.089875
9,Integrated_SoftCutoff_0.1,Fixed_0.5,0.5000,0.900646,0.885344,0.070096,0.057777,0.049629,0.492395,0.090169,0.054802,0.375744,0.089571,0.124029
10,Integrated_SoftCutoff_0.1,LOSO_Global_MacroF1,0.5270,0.900646,0.885344,0.070096,0.057777,0.051534,0.456654,0.092616,0.056359,0.344914,0.090868,0.125913
11,Integrated_SoftCutoff_0.1,LOSO_RecallPreserving_MacroF1,0.4485,0.900646,0.885344,0.070096,0.057777,0.045913,0.558723,0.084853,0.051476,0.433826,0.086167,0.119416
12,Integrated_SoftCutoff_0.2,Fixed_0.5,0.5000,0.859912,0.855595,0.025459,0.028117,0.027854,0.492727,0.052727,0.021496,0.403906,0.040212,0.058977


,model_id,mode,final_threshold_median,threshold_q1,threshold_q3,threshold_iqr,threshold_min,threshold_max,threshold_mean,threshold_sd
0,Controlled_Hard,Fixed_0.5,0.5000,0.50000,0.50000,0.00000,0.500,0.500,0.500000,0.000000
1,Controlled_Hard,LOSO_Global_MacroF1,0.6685,0.66800,0.67050,0.00250,0.621,0.837,0.689000,0.068476
2,Controlled_Hard,LOSO_RecallPreserving_MacroF1,0.5050,0.50500,0.50500,0.00000,0.505,0.507,0.505333,0.000745
6,Integrated_RawSoft,Fixed_0.5,0.5000,0.50000,0.50000,0.00000,0.500,0.500,0.500000,0.000000
7,Integrated_RawSoft,LOSO_Global_MacroF1,0.7340,0.73250,0.77300,0.04050,0.629,0.858,0.745500,0.068697
8,Integrated_RawSoft,LOSO_RecallPreserving_MacroF1,0.4870,0.47975,0.50025,0.02050,0.468,0.552,0.496000,0.027295
9,Integrated_SoftCutoff_0.1,Fixed_0.5,0.5000,0.50000,0.50000,0.00000,0.500,0.500,0.500000,0.000000
10,Integrated_SoftCutoff_0.1,LOSO_Global_MacroF1,0.5270,0.52525,0.55575,0.03050,0.525,0.790,0.576500,0.096541
11,Integrated_SoftCutoff_0.1,LOSO_RecallPreserving_MacroF1,0.4485,0.44275,0.44975,0.00700,0.424,0.457,0.444833,0.010415
12,Integrated_SoftCutoff_0.2,Fixed_0.5,0.5000,0.50000,0.50000,0.00000,0.500,0.500,0.500000,0.000000


,model_id,mode,reported_threshold,delta_vs_same_model_fixed05_micro_precision,delta_vs_same_model_fixed05_micro_recall,delta_vs_same_model_fixed05_micro_f1,delta_vs_same_model_fixed05_macro_precision,delta_vs_same_model_fixed05_macro_recall,delta_vs_same_model_fixed05_macro_f1,delta_vs_same_model_fixed05_weighted_f1,delta_vs_same_model_fixed05_true_positive_label_count,delta_vs_same_model_fixed05_false_positive_label_count,delta_vs_same_model_fixed05_false_negative_label_count,delta_vs_same_model_fixed05_predicted_positive_label_count
0,Controlled_Hard,Fixed_0.5,0.5000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0
1,Controlled_Hard,LOSO_Global_MacroF1,0.6685,0.028193,-0.162663,0.037815,0.016402,-0.142011,0.015962,0.025557,-1957.0,-64417.0,1957.0,-66374.0
2,Controlled_Hard,LOSO_RecallPreserving_MacroF1,0.5050,0.000730,-0.004073,0.001148,0.000598,-0.003799,0.000724,0.001147,-49.0,-2495.0,49.0,-2544.0
6,Integrated_RawSoft,Fixed_0.5,0.5000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0
7,Integrated_RawSoft,LOSO_Global_MacroF1,0.7340,0.027458,-0.308204,0.030304,0.024334,-0.258190,0.020766,0.032200,-3708.0,-112884.0,3708.0,-116592.0
8,Integrated_RawSoft,LOSO_RecallPreserving_MacroF1,0.4870,-0.001071,0.015959,-0.001708,-0.000795,0.015921,-0.001165,-0.001812,192.0,8697.0,-192.0,8889.0
9,Integrated_SoftCutoff_0.1,Fixed_0.5,0.5000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0
10,Integrated_SoftCutoff_0.1,LOSO_Global_MacroF1,0.5270,0.001905,-0.035741,0.002447,0.001557,-0.030830,0.001298,0.001883,-430.0,-12327.0,430.0,-12757.0
11,Integrated_SoftCutoff_0.1,LOSO_RecallPreserving_MacroF1,0.4485,-0.003716,0.066329,-0.005317,-0.003326,0.058081,-0.003404,-0.004614,798.0,26244.0,-798.0,27042.0
12,Integrated_SoftCutoff_0.2,Fixed_0.5,0.5000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0


Phase A complete. Thresholds are frozen from Validation only.
To run Phase B, set RUN_TEST_EVALUATION=True without changing the protocol.


## Phase B — Post-hoc confirmatory Test evaluation

This phase is guarded by `RUN_TEST_EVALUATION`. It reloads the frozen JSON, verifies the protocol hash, then loads Test labels/probabilities. No threshold is selected or modified using Test.

In [10]:
# =============================================================================
# Phase B: apply frozen thresholds to Test only
# =============================================================================

if not RUN_TEST_EVALUATION:
    print("TEST PHASE DISABLED. Validation thresholds are frozen; Test has not been loaded by this phase.")
else:
    assert FREEZE_PATH.exists(), "Validation threshold freeze file is missing."
    frozen_from_disk = json.loads(FREEZE_PATH.read_text(encoding="utf-8"))
    assert frozen_from_disk["protocol_sha256"] == PROTOCOL_SHA256
    assert frozen_from_disk["validation_manifest_sha256"] == VAL_MANIFEST_SHA256
    assert frozen_from_disk["frozen_thresholds"] == frozen_thresholds

    TEST_CSV = HARD_LABEL_DIR / "final_df_test.csv"
    assert TEST_CSV.exists(), TEST_CSV
    test_df = pd.read_csv(TEST_CSV, low_memory=False)
    missing_columns = [c for c in ["slide_name", "filepath", *LABEL_COLUMNS] if c not in test_df.columns]
    assert not missing_columns, f"Test CSV missing columns: {missing_columns}"
    for col in LABEL_COLUMNS:
        test_df[col] = pd.to_numeric(test_df[col], errors="raise").astype(np.int8)

    test_targets_csv = test_df[LABEL_COLUMNS].to_numpy(dtype=np.int8)
    test_slide_ids = test_df["slide_name"].astype(str).to_numpy()
    test_wsis = sorted(pd.unique(test_slide_ids).tolist())
    assert len(test_df) == EXPECTED_TEST_ROWS
    assert test_targets_csv.shape == (EXPECTED_TEST_ROWS, EXPECTED_NUM_CLASSES)
    assert len(test_wsis) == EXPECTED_TEST_WSIS
    assert set(np.unique(test_targets_csv)).issubset({0, 1})
    assert not test_df.duplicated().any()

    test_probs_by_model: Dict[str, np.ndarray] = {}
    test_input_audit_rows: List[Dict[str, Any]] = []
    for model_id in MODEL_ORDER:
        print("\n" + "=" * 100)
        print("Resolving Test probabilities:", model_id)
        probs, metadata = resolve_probabilities(
            MODEL_SPECS[model_id],
            split="test",
            df=test_df,
            expected_targets=test_targets_csv,
        )
        assert metadata["checkpoint_sha256"] == frozen_from_disk["model_checkpoint_sha256"][model_id]
        test_probs_by_model[model_id] = probs
        test_input_audit_rows.append(metadata)

    test_input_audit_df = pd.DataFrame(test_input_audit_rows)
    test_input_audit_df.to_csv(OUTPUT_DIR / "test_input_audit.csv", index=False)

    test_overall_rows: List[Dict[str, Any]] = []
    test_per_class_frames: List[pd.DataFrame] = []
    test_per_wsi_rows: List[Dict[str, Any]] = []

    for model_id in MODEL_ORDER:
        probs = test_probs_by_model[model_id]
        for mode in MODE_ORDER:
            threshold = float(frozen_from_disk["frozen_thresholds"][model_id][mode])
            pred = (probs >= threshold).astype(np.int8)
            overall = evaluate_predictions(test_targets_csv, probs, pred)
            test_overall_rows.append({
                "model_id": model_id,
                "mode": mode,
                "evaluation": "Test_PostHocConfirmatory",
                "threshold": threshold,
                **overall,
            })

            class_df = per_class_metrics(test_targets_csv, probs, pred)
            class_df.insert(0, "threshold", threshold)
            class_df.insert(0, "mode", mode)
            class_df.insert(0, "model_id", model_id)
            test_per_class_frames.append(class_df)

            for wsi in test_wsis:
                mask = test_slide_ids == wsi
                wsi_metrics = evaluate_predictions(
                    test_targets_csv[mask], probs[mask], pred[mask]
                )
                test_per_wsi_rows.append({
                    "model_id": model_id,
                    "mode": mode,
                    "wsi": wsi,
                    "rows": int(mask.sum()),
                    "threshold": threshold,
                    **wsi_metrics,
                })

    test_overall_df = pd.DataFrame(test_overall_rows)
    test_per_class_df = pd.concat(test_per_class_frames, ignore_index=True)
    test_per_wsi_df = pd.DataFrame(test_per_wsi_rows)

    test_overall_df.to_csv(OUTPUT_DIR / "test_frozen_threshold_overall.csv", index=False)
    test_per_class_df.to_csv(OUTPUT_DIR / "test_frozen_threshold_per_class.csv", index=False)
    test_per_wsi_df.to_csv(OUTPUT_DIR / "test_frozen_threshold_per_wsi.csv", index=False)

    # -------------------------------------------------------------------------
    # Fixed-0.5 regression audit against historical outputs.
    # Tolerances permit tiny metric-library differences but catch row misalignment.
    # -------------------------------------------------------------------------
    known_fixed_references = {
        "Controlled_Hard": {
            "micro_auroc": 0.915146, "macro_auroc": 0.890333,
            "micro_precision": 0.046167, "micro_recall": 0.357389, "micro_f1": 0.081771,
            "macro_precision": 0.034324, "macro_recall": 0.213731, "macro_f1": 0.055800,
            "weighted_f1": 0.135896,
        },
        "Integrated_RawSoft": {
            "micro_auroc": 0.920079, "macro_auroc": 0.896608,
            "micro_precision": 0.035084, "micro_recall": 0.269818, "micro_f1": 0.062094,
            "macro_precision": 0.028930, "macro_recall": 0.174713, "macro_f1": 0.045457,
            "weighted_f1": 0.111081,
        },
        "Integrated_SoftCutoff_0.1": {
            "micro_auroc": 0.899278, "macro_auroc": 0.887280,
            "micro_precision": 0.044345, "micro_recall": 0.306461, "micro_f1": 0.077479,
            "macro_precision": 0.037657, "macro_recall": 0.175505, "macro_f1": 0.056488,
            "weighted_f1": 0.140264,
        },
        "Integrated_SoftCutoff_0.2": {
            "micro_auroc": 0.897564, "macro_auroc": 0.863110,
            "micro_precision": 0.028685, "micro_recall": 0.243344, "micro_f1": 0.051320,
            "macro_precision": 0.026673, "macro_recall": 0.129355, "macro_f1": 0.039651,
            "weighted_f1": 0.106881,
        },
        "Integrated_SoftCutoff_0.3": {
            "micro_auroc": 0.909523, "macro_auroc": 0.863919,
            "micro_precision": 0.039040, "micro_recall": 0.252543, "micro_f1": 0.067625,
            "macro_precision": 0.032259, "macro_recall": 0.130185, "macro_f1": 0.048522,
            "weighted_f1": 0.120930,
        },
    }

    regression_rows: List[Dict[str, Any]] = []
    for model_id, expected_metrics in known_fixed_references.items():
        observed_row = test_overall_df[
            (test_overall_df["model_id"] == model_id)
            & (test_overall_df["mode"] == "Fixed_0.5")
        ].iloc[0]
        for metric, expected in expected_metrics.items():
            observed = float(observed_row[metric])
            abs_diff = abs(observed - expected)
            passed = abs_diff <= 5e-4
            regression_rows.append({
                "model_id": model_id,
                "metric": metric,
                "expected_historical": expected,
                "observed": observed,
                "absolute_difference": abs_diff,
                "pass": passed,
            })
            assert passed, (model_id, metric, expected, observed, abs_diff)

    regression_df = pd.DataFrame(regression_rows)
    regression_df.to_csv(OUTPUT_DIR / "test_fixed05_historical_regression_audit.csv", index=False)

    # -------------------------------------------------------------------------
    # Per-WSI improvement counts relative to Controlled Hard at fixed 0.5.
    # -------------------------------------------------------------------------
    baseline_wsi = test_per_wsi_df[
        (test_per_wsi_df["model_id"] == "Controlled_Hard")
        & (test_per_wsi_df["mode"] == "Fixed_0.5")
    ].set_index("wsi")

    improvement_rows: List[Dict[str, Any]] = []
    comparison_metrics = [
        "micro_f1", "macro_f1", "weighted_f1", "micro_recall", "macro_recall",
        "micro_auprc", "macro_auprc",
    ]
    for (model_id, mode), group in test_per_wsi_df.groupby(["model_id", "mode"], sort=False):
        current = group.set_index("wsi").loc[baseline_wsi.index]
        for metric in comparison_metrics:
            delta = current[metric] - baseline_wsi[metric]
            improvement_rows.append({
                "model_id": model_id,
                "mode": mode,
                "metric": metric,
                "wsi_improved": int((delta > 1e-12).sum()),
                "wsi_equal": int((delta.abs() <= 1e-12).sum()),
                "wsi_worse": int((delta < -1e-12).sum()),
                "mean_wsi_delta": float(delta.mean()),
                "median_wsi_delta": float(delta.median()),
            })

    test_wsi_improvement_df = pd.DataFrame(improvement_rows)
    test_wsi_improvement_df.to_csv(
        OUTPUT_DIR / "test_wsi_improvement_counts_vs_controlled_fixed05.csv", index=False
    )

    # -------------------------------------------------------------------------
    # Direct threshold-effect table: calibrated mode minus the same model at 0.5.
    # -------------------------------------------------------------------------
    fixed_by_model = test_overall_df[test_overall_df["mode"] == "Fixed_0.5"].set_index("model_id")
    threshold_effect_rows: List[Dict[str, Any]] = []
    for _, row in test_overall_df.iterrows():
        fixed_row = fixed_by_model.loc[row["model_id"]]
        result = {
            "model_id": row["model_id"],
            "mode": row["mode"],
            "threshold": row["threshold"],
        }
        for metric in [
            "micro_precision", "micro_recall", "micro_f1",
            "macro_precision", "macro_recall", "macro_f1", "weighted_f1",
            "true_positive_label_count", "false_positive_label_count", "false_negative_label_count",
        ]:
            result[f"delta_vs_same_model_fixed05_{metric}"] = float(row[metric]) - float(fixed_row[metric])
        threshold_effect_rows.append(result)

    threshold_effect_df = pd.DataFrame(threshold_effect_rows)
    threshold_effect_df.to_csv(OUTPUT_DIR / "test_threshold_effect_vs_same_model_fixed05.csv", index=False)

    # -------------------------------------------------------------------------
    # Predefined interpretation tags; these summarize rather than select models.
    # -------------------------------------------------------------------------
    controlled_test = test_overall_df[
        (test_overall_df["model_id"] == "Controlled_Hard")
        & (test_overall_df["mode"] == "Fixed_0.5")
    ].iloc[0]

    interpretation_rows: List[Dict[str, Any]] = []
    for model_id in MODEL_ORDER:
        for mode in ["LOSO_Global_MacroF1", "LOSO_RecallPreserving_MacroF1"]:
            row = test_overall_df[
                (test_overall_df["model_id"] == model_id)
                & (test_overall_df["mode"] == mode)
            ].iloc[0]
            recall_preserved = (
                float(row["micro_recall"]) >= float(controlled_test["micro_recall"]) - RECALL_EPSILON
                and float(row["macro_recall"]) >= float(controlled_test["macro_recall"]) - RECALL_EPSILON
            )
            both_f1_improved = (
                float(row["micro_f1"]) > float(controlled_test["micro_f1"])
                and float(row["macro_f1"]) > float(controlled_test["macro_f1"])
            )
            auprc_noninferior = (
                float(row["micro_auprc"]) >= float(controlled_test["micro_auprc"]) - 0.001
                and float(row["macro_auprc"]) >= float(controlled_test["macro_auprc"]) - 0.001
            )

            if recall_preserved and both_f1_improved and auprc_noninferior:
                tag = "RECALL_PRESERVING_SINGLE_STAGE_IMPROVEMENT_CANDIDATE"
            elif both_f1_improved and not recall_preserved:
                tag = "F1_RECOVERY_WITH_RECALL_TRADEOFF"
            else:
                tag = "NO_CONTROLLED_BASELINE_REPLACEMENT"

            interpretation_rows.append({
                "model_id": model_id,
                "mode": mode,
                "threshold": float(row["threshold"]),
                "recall_preserved_vs_controlled": recall_preserved,
                "both_micro_and_macro_f1_improved": both_f1_improved,
                "micro_and_macro_auprc_noninferior": auprc_noninferior,
                "interpretation_tag": tag,
            })

    interpretation_df = pd.DataFrame(interpretation_rows)
    interpretation_df.to_csv(OUTPUT_DIR / "test_predefined_interpretation_tags.csv", index=False)

    display_columns = [
        "model_id", "mode", "threshold",
        "micro_auroc", "macro_auroc", "micro_auprc", "macro_auprc",
        "micro_precision", "micro_recall", "micro_f1",
        "macro_precision", "macro_recall", "macro_f1", "weighted_f1",
        "true_positive_label_count", "false_positive_label_count", "false_negative_label_count",
    ]
    display(test_overall_df[display_columns].sort_values(["model_id", "mode"]))
    display(interpretation_df)
    print("PASS: Test used only with frozen Validation thresholds.")
    print("All outputs saved to:", OUTPUT_DIR)

TEST PHASE DISABLED. Validation thresholds are frozen; Test has not been loaded by this phase.


## Required reporting language

The final interpretation must separate ranking from hard decisions:

- AUROC/AUPRC differences are properties of the frozen model probabilities and cannot be caused by threshold choice.
- Precision/Recall/F1 changes between `Fixed_0.5` and calibrated modes quantify threshold mismatch and operating-point trade-offs.
- A Soft-label model should only be promoted as a single-stage improvement if its frozen Validation-selected threshold preserves the Controlled recalls and improves both Micro/Macro F1 without materially reducing AUPRC.
- Any Test result is post-hoc confirmatory because the project historically inspected this Test split.